In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_7.py ──
"""
Shared infrastructure for MLFP02 Exercise 7 — CUPED and Causal Inference.

Contains: experiment data loading, SRM check, naive A/B baseline, CUPED
math helpers, Bayesian decision utilities, mSPRT helpers, DiD scenario
simulators, and plotting helpers. Technique-specific narration and
checkpoints live in the per-technique files.

Importable from any cwd after `uv sync`:

"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

# Output directory for visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex7_causal"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Experiment data with pre-experiment covariates
# ════════════════════════════════════════════════════════════════════════


def load_experiment() -> pl.DataFrame:
    """Load the MLFP02 experiment dataset.

    Columns (required): experiment_group, revenue, pre_metric_value, timestamp
    Optional: metric_value (additional covariate for multi-CUPED)
    """
    loader = MLFPDataLoader()
    return loader.load("mlfp02", "experiment_data.parquet")


def split_groups(experiment: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Return (control, treatment) sub-frames by experiment_group column."""
    control = experiment.filter(pl.col("experiment_group") == "control")
    treatment = experiment.filter(pl.col("experiment_group") != "control")
    return control, treatment


def get_revenue_arrays(
    control: pl.DataFrame, treatment: pl.DataFrame
) -> tuple[np.ndarray, np.ndarray]:
    """Extract revenue arrays as float64 numpy arrays."""
    y_c = control["revenue"].to_numpy().astype(np.float64)
    y_t = treatment["revenue"].to_numpy().astype(np.float64)
    return y_c, y_t


def get_covariate_arrays(
    control: pl.DataFrame, treatment: pl.DataFrame, column: str = "pre_metric_value"
) -> tuple[np.ndarray, np.ndarray]:
    """Extract a pre-experiment covariate as float64 numpy arrays."""
    x_c = control[column].to_numpy().astype(np.float64)
    x_t = treatment[column].to_numpy().astype(np.float64)
    return x_c, x_t


# ════════════════════════════════════════════════════════════════════════
# SAMPLE RATIO MISMATCH (SRM)
# ════════════════════════════════════════════════════════════════════════


def compute_srm(n_c: int, n_t: int) -> float:
    """Chi-square SRM test. Returns p-value. p < 0.01 indicates SRM."""
    expected = np.array([n_c + n_t] * 2) / 2
    observed = np.array([n_c, n_t])
    _, srm_p = stats.chisquare(observed, f_exp=expected)
    return float(srm_p)


# ════════════════════════════════════════════════════════════════════════
# STANDARD A/B (BASELINE)
# ════════════════════════════════════════════════════════════════════════


def naive_ab(y_c: np.ndarray, y_t: np.ndarray) -> dict[str, float]:
    """Standard Welch-style lift, SE, 95% CI, z, two-sided p-value."""
    n_c, n_t = len(y_c), len(y_t)
    mean_c, mean_t = y_c.mean(), y_t.mean()
    lift = mean_t - mean_c
    se = float(np.sqrt(y_c.var(ddof=1) / n_c + y_t.var(ddof=1) / n_t))
    ci_lo = lift - 1.96 * se
    ci_hi = lift + 1.96 * se
    z = lift / se if se > 0 else 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {
        "mean_c": float(mean_c),
        "mean_t": float(mean_t),
        "lift": float(lift),
        "se": se,
        "ci_lo": float(ci_lo),
        "ci_hi": float(ci_hi),
        "z": float(z),
        "p_value": float(p),
    }


# ════════════════════════════════════════════════════════════════════════
# CUPED — SINGLE COVARIATE
# ════════════════════════════════════════════════════════════════════════


def single_cov_cuped(
    y_c: np.ndarray, y_t: np.ndarray, x_c: np.ndarray, x_t: np.ndarray
) -> dict[str, Any]:
    """Single-covariate CUPED: Y_adj = Y - theta*(X - E[X]).

    theta = Cov(Y, X) / Var(X). Returns adjusted arrays, point estimate,
    SE, CI, p-value, theta, rho, and variance reduction.
    """
    x_all = np.concatenate([x_c, x_t])
    y_all = np.concatenate([y_c, y_t])
    var_x = np.var(x_all, ddof=1)
    theta = np.cov(y_all, x_all)[0, 1] / var_x if var_x > 0 else 0.0
    rho = np.corrcoef(y_all, x_all)[0, 1]
    x_mean = x_all.mean()

    y_c_adj = y_c - theta * (x_c - x_mean)
    y_t_adj = y_t - theta * (x_t - x_mean)

    n_c, n_t = len(y_c), len(y_t)
    lift_adj = y_t_adj.mean() - y_c_adj.mean()
    se = float(np.sqrt(y_c_adj.var(ddof=1) / n_c + y_t_adj.var(ddof=1) / n_t))
    ci_lo, ci_hi = lift_adj - 1.96 * se, lift_adj + 1.96 * se
    z = lift_adj / se if se > 0 else 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))

    return {
        "theta": float(theta),
        "rho": float(rho),
        "y_c_adj": y_c_adj,
        "y_t_adj": y_t_adj,
        "lift": float(lift_adj),
        "se": se,
        "ci_lo": float(ci_lo),
        "ci_hi": float(ci_hi),
        "z": float(z),
        "p_value": float(p),
        "theoretical_reduction": float(rho**2),
    }


# ════════════════════════════════════════════════════════════════════════
# CUPED — MULTI-COVARIATE
# ════════════════════════════════════════════════════════════════════════


def multi_cov_cuped(
    y_c: np.ndarray,
    y_t: np.ndarray,
    X_c: np.ndarray,
    X_t: np.ndarray,
) -> dict[str, Any]:
    """Multi-covariate CUPED via OLS on centered covariates.

    theta = (X'X)^-1 X'Y — multivariate regression coefficients.
    """
    y_all = np.concatenate([y_c, y_t])
    X_all = np.vstack([X_c, X_t])
    X_mean = X_all.mean(axis=0)
    X_centered = X_all - X_mean
    theta = np.linalg.lstsq(X_centered, y_all - y_all.mean(), rcond=None)[0]

    y_c_adj = y_c - (X_c - X_mean) @ theta
    y_t_adj = y_t - (X_t - X_mean) @ theta

    n_c, n_t = len(y_c), len(y_t)
    lift = y_t_adj.mean() - y_c_adj.mean()
    se = float(np.sqrt(y_c_adj.var(ddof=1) / n_c + y_t_adj.var(ddof=1) / n_t))
    return {
        "theta": theta,
        "y_c_adj": y_c_adj,
        "y_t_adj": y_t_adj,
        "lift": float(lift),
        "se": se,
        "ci_lo": float(lift - 1.96 * se),
        "ci_hi": float(lift + 1.96 * se),
    }


# ════════════════════════════════════════════════════════════════════════
# CUPED — STRATIFIED
# ════════════════════════════════════════════════════════════════════════


def stratify_by_covariate(
    x_c: np.ndarray, x_t: np.ndarray, percentiles: tuple[int, int] = (33, 67)
) -> dict[str, np.ndarray]:
    """Build Low/Medium/High strata masks over concatenated (x_c, x_t)."""
    x_all = np.concatenate([x_c, x_t])
    q_lo, q_hi = np.percentile(x_all, percentiles)
    return {
        "Low spenders": (x_all <= q_lo),
        "Medium spenders": (x_all > q_lo) & (x_all <= q_hi),
        "High spenders": (x_all > q_hi),
    }


def stratified_cuped(
    y_c: np.ndarray,
    y_t: np.ndarray,
    x_c: np.ndarray,
    x_t: np.ndarray,
    strata: dict[str, np.ndarray],
    min_per_cell: int = 30,
) -> dict[str, dict[str, float]]:
    """Apply CUPED within each stratum. Returns {name: {n_ctrl,n_treat,lift,se,p}}."""
    n_c = len(y_c)
    results: dict[str, dict[str, float]] = {}
    for name, mask in strata.items():
        ctrl_mask, treat_mask = mask[:n_c], mask[n_c:]
        y_c_s, y_t_s = y_c[ctrl_mask], y_t[treat_mask]
        x_c_s, x_t_s = x_c[ctrl_mask], x_t[treat_mask]
        if len(y_c_s) < min_per_cell or len(y_t_s) < min_per_cell:
            continue
        x_all_s = np.concatenate([x_c_s, x_t_s])
        y_all_s = np.concatenate([y_c_s, y_t_s])
        var_x = np.var(x_all_s, ddof=1)
        theta_s = np.cov(y_all_s, x_all_s)[0, 1] / var_x if var_x > 0 else 0.0
        x_mean = x_all_s.mean()
        y_c_adj = y_c_s - theta_s * (x_c_s - x_mean)
        y_t_adj = y_t_s - theta_s * (x_t_s - x_mean)
        lift = y_t_adj.mean() - y_c_adj.mean()
        se = float(
            np.sqrt(y_c_adj.var(ddof=1) / len(y_c_s) + y_t_adj.var(ddof=1) / len(y_t_s))
        )
        z = lift / se if se > 0 else 0.0
        p = 2 * (1 - stats.norm.cdf(abs(z)))
        results[name] = {
            "n_ctrl": len(y_c_s),
            "n_treat": len(y_t_s),
            "lift": float(lift),
            "se": se,
            "p_value": float(p),
        }
    return results


# ════════════════════════════════════════════════════════════════════════
# BAYESIAN A/B
# ════════════════════════════════════════════════════════════════════════


def bayesian_decision(
    y_c_adj: np.ndarray,
    y_t_adj: np.ndarray,
    lift: float,
    practical_threshold: float = 1.0,
) -> dict[str, float]:
    """Bayesian posterior using normal approximation on CUPED-adjusted arrays.

    Returns P(treatment > control), P(treatment > control + threshold),
    expected loss (both directions), and credible interval.
    """
    n_c, n_t = len(y_c_adj), len(y_t_adj)
    se_c = y_c_adj.std(ddof=1) / np.sqrt(n_c)
    se_t = y_t_adj.std(ddof=1) / np.sqrt(n_t)
    se_lift = float(np.sqrt(se_c**2 + se_t**2))

    prob_better = float(1 - stats.norm.cdf(0, loc=lift, scale=se_lift))
    prob_practical = float(
        1 - stats.norm.cdf(practical_threshold, loc=lift, scale=se_lift)
    )

    z = -lift / se_lift if se_lift > 0 else 0.0
    exp_loss_treat = float(se_lift * stats.norm.pdf(z) + lift * stats.norm.cdf(z))
    exp_loss_ctrl = float(se_lift * stats.norm.pdf(-z) - lift * stats.norm.cdf(-z))

    return {
        "prob_treatment_better": prob_better,
        "prob_practical": prob_practical,
        "expected_loss_treatment": max(0.0, exp_loss_treat),
        "expected_loss_control": max(0.0, exp_loss_ctrl),
        "se_lift": se_lift,
        "ci_lo": float(lift - 1.96 * se_lift),
        "ci_hi": float(lift + 1.96 * se_lift),
    }


def bayesian_decision_rule(prob_better: float, exp_loss_treat: float) -> str:
    """Simple ship/continue/hold decision rule."""
    if prob_better > 0.95 and exp_loss_treat < 0.50:
        return "SHIP — high confidence + low expected loss"
    if prob_better > 0.80:
        return "CONTINUE — promising but need more data"
    return "HOLD — insufficient evidence"


# ════════════════════════════════════════════════════════════════════════
# SEQUENTIAL TESTING — mSPRT
# ════════════════════════════════════════════════════════════════════════


def msprt_sequential_pvalues(
    experiment: pl.DataFrame,
    tau_sq: float,
    min_per_group: int = 100,
    skip_first_days: int = 3,
) -> list[dict[str, float]]:
    """Walk the experiment day by day, computing fixed and mSPRT p-values.

    tau_sq is the mSPRT hyperparameter — typically set to se_naive**2.
    """
    if experiment["timestamp"].dtype in [pl.Utf8, pl.String]:
        exp_daily = experiment.with_columns(
            pl.col("timestamp")
            .str.to_datetime("%Y-%m-%d %H:%M:%S")
            .dt.date()
            .alias("day")
        )
    else:
        exp_daily = experiment.with_columns(
            pl.col("timestamp").cast(pl.Date).alias("day")
        )

    days = sorted(exp_daily["day"].unique().to_list())
    results: list[dict[str, float]] = []
    for i, day in enumerate(days):
        if i < skip_first_days:
            continue
        cumulative = exp_daily.filter(pl.col("day") <= day)
        c = (
            cumulative.filter(pl.col("experiment_group") == "control")["revenue"]
            .to_numpy()
            .astype(np.float64)
        )
        t = (
            cumulative.filter(pl.col("experiment_group") != "control")["revenue"]
            .to_numpy()
            .astype(np.float64)
        )
        if len(c) < min_per_group or len(t) < min_per_group:
            continue
        diff = t.mean() - c.mean()
        v_n = c.var(ddof=1) / len(c) + t.var(ddof=1) / len(t)
        se = float(np.sqrt(v_n))
        z = diff / se if se > 0 else 0.0
        p_fixed = float(2 * (1 - stats.norm.cdf(abs(z))))
        # mSPRT always-valid p-value
        lambda_n = np.sqrt(v_n / (v_n + tau_sq)) * np.exp(
            tau_sq * z**2 / (2 * (v_n + tau_sq))
        )
        p_seq = float(min(1.0, 1.0 / lambda_n)) if lambda_n > 0 else 1.0
        results.append(
            {
                "day": i + 1,
                "n": int(len(c) + len(t)),
                "lift": float(diff),
                "p_fixed": p_fixed,
                "p_sequential": p_seq,
            }
        )
    return results


# ════════════════════════════════════════════════════════════════════════
# PEEKING PROBLEM SIMULATION
# ════════════════════════════════════════════════════════════════════════


def simulate_peeking(
    n_sims: int = 1000,
    n_per_sim: int = 2000,
    n_checks: int = 20,
    seed: int = 42,
) -> dict[str, float]:
    """Simulate A/A experiments (zero effect) with and without peeking.

    Returns dict with false-positive rates for: no-peek, fixed-p peeking,
    plus the theoretical peeking inflation for comparison.
    """
    rng = np.random.default_rng(seed=seed)
    false_pos_fixed = 0
    false_pos_no_peek = 0

    for _ in range(n_sims):
        sim_ctrl = rng.normal(50, 10, size=n_per_sim)
        sim_treat = rng.normal(50, 10, size=n_per_sim)  # no real effect

        # no peeking
        z_end = (sim_treat.mean() - sim_ctrl.mean()) / np.sqrt(
            sim_ctrl.var(ddof=1) / n_per_sim + sim_treat.var(ddof=1) / n_per_sim
        )
        if 2 * (1 - stats.norm.cdf(abs(z_end))) < 0.05:
            false_pos_no_peek += 1

        # peeking at n_checks points, fixed p-values
        peeked_sig = False
        for check_n in np.linspace(100, n_per_sim, n_checks, dtype=int):
            sc = sim_ctrl[:check_n]
            st = sim_treat[:check_n]
            se_p = np.sqrt(sc.var(ddof=1) / check_n + st.var(ddof=1) / check_n)
            z_p = (st.mean() - sc.mean()) / se_p if se_p > 0 else 0
            if 2 * (1 - stats.norm.cdf(abs(z_p))) < 0.05:
                peeked_sig = True
                break
        if peeked_sig:
            false_pos_fixed += 1

    return {
        "n_sims": n_sims,
        "n_checks": n_checks,
        "rate_no_peek": false_pos_no_peek / n_sims,
        "rate_fixed_peek": false_pos_fixed / n_sims,
        "theoretical_inflated_rate": 1 - (1 - 0.05) ** n_checks,
    }


# ════════════════════════════════════════════════════════════════════════
# DIFFERENCE-IN-DIFFERENCES — Singapore HDB cooling measures
# ════════════════════════════════════════════════════════════════════════


def simulate_hdb_cooling_measures(
    n_per_cell: int = 500, seed: int = 99
) -> dict[str, np.ndarray]:
    """Simulate Singapore HDB prices around a stamp-duty cooling measure.

    Treatment group: Central area HDB transactions (hit by the policy).
    Control group: Non-Central area transactions (exempt).

    Returns the four cells as arrays: pre_central, post_central,
    pre_noncentral, post_noncentral.
    """
    rng = np.random.default_rng(seed=seed)
    pre_central = rng.normal(550_000, 80_000, size=n_per_cell)
    pre_noncentral = rng.normal(450_000, 70_000, size=n_per_cell)
    # Policy effect: Central drops $20K; both grow $10K baseline.
    post_central = rng.normal(540_000, 85_000, size=n_per_cell)
    post_noncentral = rng.normal(460_000, 72_000, size=n_per_cell)
    return {
        "pre_central": pre_central,
        "post_central": post_central,
        "pre_noncentral": pre_noncentral,
        "post_noncentral": post_noncentral,
    }


def diff_in_diff(cells: dict[str, np.ndarray]) -> dict[str, float]:
    """Compute DiD estimate, SE, CI, z, p from four cell arrays."""
    y_tp = cells["pre_central"].mean()
    y_tq = cells["post_central"].mean()
    y_cp = cells["pre_noncentral"].mean()
    y_cq = cells["post_noncentral"].mean()

    did = (y_tq - y_tp) - (y_cq - y_cp)
    n = len(cells["pre_central"])
    se = float(
        np.sqrt(
            cells["pre_central"].var(ddof=1) / n
            + cells["post_central"].var(ddof=1) / n
            + cells["pre_noncentral"].var(ddof=1) / n
            + cells["post_noncentral"].var(ddof=1) / n
        )
    )
    z = did / se if se > 0 else 0.0
    p = float(2 * (1 - stats.norm.cdf(abs(z))))
    return {
        "y_treat_pre": float(y_tp),
        "y_treat_post": float(y_tq),
        "y_ctrl_pre": float(y_cp),
        "y_ctrl_post": float(y_cq),
        "did_estimate": float(did),
        "se": se,
        "ci_lo": float(did - 1.96 * se),
        "ci_hi": float(did + 1.96 * se),
        "z": float(z),
        "p_value": p,
    }


# ════════════════════════════════════════════════════════════════════════
# PARALLEL TRENDS TEST
# ════════════════════════════════════════════════════════════════════════


def parallel_trends_test(
    n_periods: int = 6,
    n_per_period: int = 200,
    central_base: float = 530_000,
    noncentral_base: float = 430_000,
    growth_per_period: float = 2000,
    seed: int = 99,
) -> dict[str, Any]:
    """Simulate pre-period trends and run a bootstrap test for parallel slopes.

    Returns pre-period means, slopes, slope difference, bootstrap p-value,
    and a pass/fail flag at alpha=0.05.
    """
    rng = np.random.default_rng(seed=seed)
    pre_central, pre_noncentral = [], []
    for t in range(n_periods):
        c = rng.normal(central_base + t * growth_per_period, 80_000, size=n_per_period)
        nc = rng.normal(
            noncentral_base + t * growth_per_period, 70_000, size=n_per_period
        )
        pre_central.append(c.mean())
        pre_noncentral.append(nc.mean())
    time_points = np.arange(n_periods)
    slope_c = float(np.polyfit(time_points, pre_central, 1)[0])
    slope_nc = float(np.polyfit(time_points, pre_noncentral, 1)[0])
    slope_diff = slope_c - slope_nc

    # Bootstrap
    n_boot = 5000
    boot_diffs = []
    for _ in range(n_boot):
        noise_c = rng.normal(0, 1000, size=n_periods)
        noise_nc = rng.normal(0, 1000, size=n_periods)
        s_c = np.polyfit(time_points, np.array(pre_central) + noise_c, 1)[0]
        s_nc = np.polyfit(time_points, np.array(pre_noncentral) + noise_nc, 1)[0]
        boot_diffs.append(s_c - s_nc)
    boot_p = float(np.mean(np.abs(boot_diffs) >= np.abs(slope_diff)))

    return {
        "pre_central": pre_central,
        "pre_noncentral": pre_noncentral,
        "time_points": time_points,
        "slope_central": slope_c,
        "slope_noncentral": slope_nc,
        "slope_diff": float(slope_diff),
        "bootstrap_p": boot_p,
        "passes": boot_p > 0.05,
    }


# ════════════════════════════════════════════════════════════════════════
# VARIANCE REDUCTION REPORTING
# ════════════════════════════════════════════════════════════════════════


def variance_reduction(se_baseline: float, se_adjusted: float) -> dict[str, float]:
    """Report variance and CI-width reduction from baseline -> adjusted SE."""
    var_red = 1 - (se_adjusted**2) / (se_baseline**2)
    ci_red = 1 - (se_adjusted / se_baseline)
    return {
        "variance_reduction": float(var_red),
        "ci_width_reduction": float(ci_red),
        "effective_sample_multiplier": float(1 / max(1 - var_red, 1e-9)),
    }


def print_banner(title: str) -> None:
    """Consistent section header across technique files."""
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 7.1: CUPED Variance Reduction
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Derive and implement CUPED using pre-experiment covariates
#   - Quantify variance reduction from the pre-post correlation (rho^2)
#   - Extend CUPED to multiple covariates via multivariate regression
#   - Apply stratified CUPED to detect heterogeneous treatment effects
#   - Log results to ExperimentTracker for reproducibility
#
# PREREQUISITES: Exercises 3-4 — hypothesis testing, p-values, SRM
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Load experiment data, SRM check
#   2. Standard A/B baseline (no CUPED)
#   3. Single-covariate CUPED: derive theta, adjust Y, verify reduction
#   4. Multi-covariate CUPED: multivariate regression
#   5. Stratified CUPED: segment-level treatment effects
#   6. Visualise and log results
#
# THEORY (CUPED):
#   Y_adj = Y - theta*(X - E[X])  where theta = Cov(Y,X)/Var(X)
#   Var(Y_adj) = Var(Y)(1 - rho^2) where rho = Cor(Y,X)
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import plotly.graph_objects as go
from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker



## THEORY — Why Variance Reduction Matters for A/B Tests


In [ ]:
# An A/B test's power depends on the standard error of the treatment
# effect estimate. Smaller SE = narrower CI = faster decisions.
#
# CUPED (Controlled-experiment Using Pre-Experiment Data) exploits the
# correlation between pre-experiment behaviour (X) and the outcome (Y).
# If a user spent $100 before the experiment, they will probably spend
# around $100 during — this predictable portion is noise we can remove.
#
# Y_adj = Y - theta*(X - E[X])
# theta = Cov(Y, X) / Var(X)  — the optimal noise-removal coefficient
# Var(Y_adj) = Var(Y) * (1 - rho^2)
#
# With rho = 0.7, CUPED removes 49% of variance — equivalent to
# doubling your sample size for free!
#
# WHY THIS MATTERS: At Shopee (Singapore), CUPED reduced experiment
# duration from 14 days to 7 days by shrinking confidence intervals —
# letting product teams ship features twice as fast.



## TASK 1 — Load Data and SRM Check


In [ ]:
print_banner("MLFP02 Exercise 7.1: CUPED Variance Reduction")

# TODO: Load the experiment dataset using the shared helper.
# Hint: load_experiment() returns a polars DataFrame.
experiment = ____

print(f"\n  Data loaded: experiment_data.parquet")
print(f"  Shape: {experiment.shape}")
print(f"  Columns: {experiment.columns}")
print(experiment.head(5))

# TODO: Split into control and treatment groups using the shared helper.
# Hint: split_groups(experiment) returns (control, treatment).
control, treatment = ____

n_c, n_t = control.height, treatment.height
print(f"\nControl: {n_c:,} | Treatment: {n_t:,}")

# TODO: Compute the SRM p-value using the shared helper.
# Hint: compute_srm(n_c, n_t) returns a float p-value.
srm_p = ____
print(f"\nSRM check: p={srm_p:.6f} — {'OK' if srm_p > 0.01 else 'SRM DETECTED'}")

# Explore pre-experiment covariates
for col in ["revenue", "pre_metric_value", "metric_value"]:
    if col in experiment.columns:
        vals = experiment[col].drop_nulls()
        print(f"  {col}: mean={vals.mean():.2f}, std={vals.std():.2f}, n={vals.len()}")



### Checkpoint 1


In [ ]:
assert 0 <= srm_p <= 1, "SRM p-value must be valid"
print("\n>>> Checkpoint 1 passed -- SRM check and data exploration completed\n")



## TASK 2 — Standard Analysis Baseline (No CUPED)


In [ ]:
# TODO: Extract revenue arrays from control and treatment.
# Hint: get_revenue_arrays(control, treatment) returns (y_c, y_t).
y_c, y_t = ____

# TODO: Run the naive A/B analysis using the shared helper.
# Hint: naive_ab(y_c, y_t) returns a dict with lift, se, ci_lo, ci_hi, p_value.
baseline = ____

print(f"\n=== Standard Analysis (no CUPED) ===")
print(f"Control mean: ${baseline['mean_c']:.2f}")
print(f"Treatment mean: ${baseline['mean_t']:.2f}")
print(
    f"Lift: ${baseline['lift']:.2f} ({baseline['lift'] / baseline['mean_c']:.2%} relative)"
)
print(f"SE: ${baseline['se']:.2f}")
print(f"95% CI: [${baseline['ci_lo']:.2f}, ${baseline['ci_hi']:.2f}]")
print(f"CI width: ${baseline['ci_hi'] - baseline['ci_lo']:.2f}")
print(f"p-value: {baseline['p_value']:.6f}")
# INTERPRETATION: The naive analysis uses only experiment-period data.
# It ignores that some users are naturally high-spenders — CUPED
# removes this baseline noise by leveraging pre-experiment data.



### Checkpoint 2


In [ ]:
assert baseline["se"] > 0, "SE must be positive"
assert baseline["ci_lo"] < baseline["ci_hi"], "CI lower must be below upper"
print("\n>>> Checkpoint 2 passed -- standard analysis baseline established\n")



## TASK 3 — Single-Covariate CUPED


In [ ]:
# CUPED: Y_adj = Y - theta*(X - E[X])
# theta = Cov(Y, X) / Var(X) — the optimal coefficient
# Var(Y_adj) = Var(Y)(1 - rho^2) where rho = Cor(Y, X)

# TODO: Extract the pre-experiment covariate arrays.
# Hint: get_covariate_arrays(control, treatment) returns (x_c, x_t).
x_c, x_t = ____

# TODO: Apply single-covariate CUPED using the shared helper.
# Hint: single_cov_cuped(y_c, y_t, x_c, x_t) returns a dict with
#   theta, rho, y_c_adj, y_t_adj, lift, se, ci_lo, ci_hi, p_value,
#   theoretical_reduction.
cuped = ____

# TODO: Compute the variance reduction metrics.
# Hint: variance_reduction(baseline_se, cuped_se) returns a dict with
#   variance_reduction, ci_width_reduction, effective_sample_multiplier.
vr = ____

print(f"\n=== Single-Covariate CUPED ===")
print(f"Pre-post correlation (rho): {cuped['rho']:.4f}")
print(f"theta (optimal coefficient): {cuped['theta']:.4f}")
print(f"Theoretical variance reduction: {cuped['theoretical_reduction']:.1%}")
print(f"Actual variance reduction: {vr['variance_reduction']:.1%}")
print(f"CI width reduction: {vr['ci_width_reduction']:.1%}")
print(f"\nCUPED lift: ${cuped['lift']:.2f}")
print(f"SE (naive): ${baseline['se']:.2f} -> SE (CUPED): ${cuped['se']:.2f}")
ci_w_naive = baseline["ci_hi"] - baseline["ci_lo"]
ci_w_cuped = cuped["ci_hi"] - cuped["ci_lo"]
print(f"CI width (naive): ${ci_w_naive:.2f} -> CI width (CUPED): ${ci_w_cuped:.2f}")
print(f"95% CI: [${cuped['ci_lo']:.2f}, ${cuped['ci_hi']:.2f}]")
print(f"p-value: {cuped['p_value']:.6f} (was {baseline['p_value']:.6f})")

# Verify CUPED does not bias the point estimate
print(f"\n--- Bias Check ---")
print(f"Naive lift: ${baseline['lift']:.4f}")
print(f"CUPED lift: ${cuped['lift']:.4f}")
print(f"Difference: ${cuped['lift'] - baseline['lift']:.4f}")
print(
    f"CUPED is {'unbiased' if abs(cuped['lift'] - baseline['lift']) < 2 * baseline['se'] else 'BIASED -- investigate'}"
)



### Checkpoint 3


In [ ]:
assert 0 <= abs(cuped["rho"]) <= 1, "Correlation must be between -1 and 1"
assert cuped["se"] <= baseline["se"] * 1.01, "CUPED SE must be <= naive SE"
assert (
    abs(vr["variance_reduction"] - cuped["theoretical_reduction"]) < 0.1
), "Actual reduction should approximate theoretical rho^2"
print("\n>>> Checkpoint 3 passed -- CUPED variance reduction verified\n")



## TASK 4 — Multi-Covariate CUPED


In [ ]:
# With multiple pre-experiment features, use multivariate regression
# to compute the optimal adjustment: Y_adj = Y - X*theta where
# theta = (X'X)^-1 X'Y (regression coefficients from Y on pre-covariates)

print(f"\n=== Multi-Covariate CUPED ===")

pre_features = ["pre_metric_value"]
if "metric_value" in control.columns:
    pre_features.append("metric_value")

X_c_multi = control.select(pre_features).to_numpy().astype(np.float64)
X_t_multi = treatment.select(pre_features).to_numpy().astype(np.float64)

# TODO: Apply multi-covariate CUPED using the shared helper.
# Hint: multi_cov_cuped(y_c, y_t, X_c_multi, X_t_multi)
multi = ____
vr_multi = variance_reduction(baseline["se"], multi["se"])

print(f"Covariates: {pre_features}")
print(f"Multi-covariate theta: {multi['theta']}")
print(
    f"Variance reduction: {vr_multi['variance_reduction']:.1%} (single-cov: {vr['variance_reduction']:.1%})"
)
print(
    f"SE: ${multi['se']:.2f} (single: ${cuped['se']:.2f}, naive: ${baseline['se']:.2f})"
)
print(f"CI: [${multi['ci_lo']:.2f}, ${multi['ci_hi']:.2f}]")



### Checkpoint 4


In [ ]:
assert multi["se"] <= baseline["se"] * 1.01, "Multi-CUPED SE should be <= naive"
print("\n>>> Checkpoint 4 passed -- multi-covariate CUPED completed\n")



## TASK 5 — Stratified CUPED: Heterogeneous Treatment Effects


In [ ]:
# Do different user segments respond differently to treatment?

print(f"\n=== Stratified CUPED ===")

# TODO: Build strata masks from the covariate arrays.
# Hint: stratify_by_covariate(x_c, x_t) returns a dict of boolean masks.
strata = ____

# TODO: Run stratified CUPED using the shared helper.
# Hint: stratified_cuped(y_c, y_t, x_c, x_t, strata)
strat_results = ____

print(
    f"{'Stratum':<20} {'n_ctrl':>8} {'n_treat':>8} {'Lift':>10} {'SE':>8} {'p-value':>10}"
)
print("-" * 68)
for name, r in strat_results.items():
    print(
        f"{name:<20} {r['n_ctrl']:>8,} {r['n_treat']:>8,} "
        f"${r['lift']:>8.2f} ${r['se']:>6.2f} {r['p_value']:>10.6f}"
    )



### Checkpoint 5


In [ ]:
assert len(strat_results) >= 2, "Should have at least 2 strata"
print("\n>>> Checkpoint 5 passed -- stratified CUPED completed\n")



## TASK 6 — Visualise: Naive vs CUPED Confidence Intervals


In [ ]:
# TODO: Create a plotly figure comparing CIs for Naive, CUPED (1-cov),
# and CUPED (multi). Each method gets a horizontal line with markers at
# (ci_lo, lift, ci_hi).
# Hint: Use go.Scatter with x=[lo, lift, hi], y=[method_name]*3
fig = go.Figure()
methods = ["Naive", "CUPED (1-cov)", "CUPED (multi)"]
ses = [baseline["se"], cuped["se"], multi["se"]]
lifts = [baseline["lift"], cuped["lift"], multi["lift"]]
for m, s, l in zip(methods, ses, lifts):
    lo, hi = l - 1.96 * s, l + 1.96 * s
    fig.add_trace(
        go.Scatter(
            x=[lo, l, hi],
            y=[m] * 3,
            mode="markers+lines",
            name=m,
            marker={"size": [8, 12, 8]},
        )
    )
fig.add_vline(x=0, line_dash="dot", line_color="red")
fig.update_layout(
    title="Confidence Intervals: Naive vs CUPED",
    xaxis_title="Treatment Effect ($)",
)
out_path = OUTPUT_DIR / "cuped_comparison.html"
fig.write_html(str(out_path))
print(f"\nSaved: {out_path}")



## APPLY — Shopee Singapore: Faster Experiment Decisions


In [ ]:
# Scenario: Shopee's experimentation platform runs ~200 A/B tests per
# quarter. Typical experiment duration is 14 days with standard analysis.
#
# With CUPED (rho ~0.7), the variance reduction of ~49% means confidence
# intervals shrink by ~30%. This lets teams reach statistical significance
# in ~7 days instead of 14 — effectively doubling experiment throughput.

print(f"\n--- Singapore Application: E-Commerce Experiment Velocity ---")
eff_mult = vr["effective_sample_multiplier"]
speedup = 1 - 1 / eff_mult if eff_mult > 1 else 0
print(f"Effective sample multiplier: {eff_mult:.1f}x")
print(f"Experiment duration reduction: {speedup:.0%}")
print(f"If base duration is 14 days -> CUPED duration: {14 / eff_mult:.0f} days")
print(f"Quarterly experiments (200 base): {200 * eff_mult:.0f} with CUPED")
savings = 200 * speedup * 50_000
print(f"Estimated annual savings: S${savings:,.0f}")



## LOG — ExperimentTracker


In [ ]:
async def log_cuped_results():
    db = "sqlite:///mlfp02_experiments.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(db)
    await conn.initialize()

    exp_id = "mlfp02_ex7_cuped"

    async with tracker.track(experiment=exp_id, run_name="cuped_analysis") as run:
        await run.log_params(
            {
                "cuped_covariate": "pre_metric_value",
                "cuped_theta": str(float(cuped["theta"])),
                "cuped_rho": str(float(cuped["rho"])),
                "n_covariates_multi": str(len(pre_features)),
            }
        )
        await run.log_metrics(
            {
                "lift_naive": float(baseline["lift"]),
                "lift_cuped": float(cuped["lift"]),
                "se_naive": float(baseline["se"]),
                "se_cuped": float(cuped["se"]),
                "variance_reduction": float(vr["variance_reduction"]),
                "ci_width_reduction": float(vr["ci_width_reduction"]),
                "p_naive": float(baseline["p_value"]),
                "p_cuped": float(cuped["p_value"]),
            }
        )
    print(f"\nLogged CUPED experiment run")
    await conn.close()


try:
    await log_cuped_results()
except Exception as e:
    print(f"  [Skipped: ExperimentTracker logging ({type(e).__name__}: {e})]")



### Checkpoint 6


In [ ]:
print("\n>>> Checkpoint 6 passed -- visualisation and logging complete\n")



## REFLECTION


- CUPED: Y_adj = Y - theta*(X - E[X]), theta = Cov(Y,X)/Var(X)
  - Variance reduction: Var(Y_adj) = Var(Y)(1 - rho^2)
  - Multi-covariate CUPED: multivariate regression for adjustment
  - Stratified CUPED: heterogeneous treatment effects by segment
  - Bias check: CUPED preserves the point estimate, only reduces SE

  NEXT: In 02_bayesian_ab.py, you'll learn to compute P(B > A)
  and make ship/continue/hold decisions using expected loss.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print("\n>>> Exercise 7.1 complete -- CUPED Variance Reduction")

